# Estimate MAG abundance from ASV abundance

This notebook is the execution workflow for `mag_abundance.py`.

The estimator itself is strict and does not silently intersect ASVs, remove missing samples/time points, or subset MAGs. The copy-number matrix supplied here should already contain the intended candidate MAG set.


## 1. Set file paths

Edit these paths and run the notebook from the repository root.


In [1]:
from pathlib import Path

COPY_NUMBER_CSV = Path("copy_number_matrix.csv")
ASV_ABUNDANCE_CSV = Path("asv_abundance.csv")
OUTPUT_CSV = Path("mag_abundance.csv")


## 2. Load and validate input tables


In [2]:
import numpy as np
import pandas as pd

from mag_abundance import estimate_mag_abundance

C_full = pd.read_csv(COPY_NUMBER_CSV, index_col=0)
Y_full = pd.read_csv(ASV_ABUNDANCE_CSV, index_col=0)

for name, frame in (("copy-number matrix", C_full), ("ASV-abundance matrix", Y_full)):
    if not frame.index.is_unique:
        raise ValueError(f"Duplicate row IDs in {name}.")
    if not frame.columns.is_unique:
        raise ValueError(f"Duplicate column IDs in {name}.")
    if frame.index.to_frame(index=False).isna().to_numpy().any():
        raise ValueError(f"Missing row IDs in {name}.")
    if frame.columns.to_frame(index=False).isna().to_numpy().any():
        raise ValueError(f"Missing column IDs in {name}.")

if any(pd.api.types.is_complex_dtype(dtype) for dtype in C_full.dtypes):
    raise ValueError("copy-number matrix must contain real values.")
if any(pd.api.types.is_complex_dtype(dtype) for dtype in Y_full.dtypes):
    raise ValueError("ASV-abundance matrix must contain real values.")

try:
    C_values = C_full.to_numpy(dtype=float)
    Y_values = Y_full.to_numpy(dtype=float)
except (TypeError, ValueError) as exc:
    raise ValueError("Both input tables must contain numeric values.") from exc

if not np.isfinite(C_values).all():
    raise ValueError("copy-number matrix contains NaN or infinity.")
if (C_values < 0).any() or not np.equal(C_values, np.floor(C_values)).all():
    raise ValueError("copy-number matrix must contain nonnegative integer copy numbers.")

# NaN is allowed here only because completely missing columns are handled below.
if np.isinf(Y_values).any():
    raise ValueError("ASV-abundance matrix contains infinity.")
if (Y_values < 0).any():
    raise ValueError("ASV-abundance matrix contains negative values.")

# Use the validated numeric representations for all subsequent processing.
C_full = pd.DataFrame(C_values, index=C_full.index, columns=C_full.columns)
Y_full = pd.DataFrame(Y_values, index=Y_full.index, columns=Y_full.columns)

print(f"copy-number matrix: {C_full.shape[0]} ASVs × {C_full.shape[1]} MAGs")
print(f"ASV-abundance matrix: {Y_full.shape[0]} ASVs × {Y_full.shape[1]} samples/time points")
display(C_full)
display(Y_full)


copy-number matrix: 2 ASVs × 3 MAGs
ASV-abundance matrix: 2 ASVs × 5 samples/time points


,MAG1,MAG2,MAG3
ASV,,,
ASV1,2.0,0.0,3.0
ASV2,0.0,4.0,2.0


,T1,T2,T3,T4,T5
ASV,,,,,
ASV1,41.0,82.0,123.0,NaN,164.0
ASV2,22.0,44.0,66.0,NaN,88.0


## 3. Match ASVs explicitly

ASVs that map to none of the candidate MAGs are removed first. The remaining ASVs are intersected with the abundance table. The counts below make every removal visible.


In [3]:
mapped_to_candidate_mag = C_full.ne(0).any(axis=1)
C_mapped = C_full.loc[mapped_to_candidate_mag].copy()

# Preserve the copy-number-matrix row order.
common_asvs = C_mapped.index[C_mapped.index.isin(Y_full.index)]
if len(common_asvs) == 0:
    raise ValueError("No common ASVs remain after matching the two input tables.")

print(f"ASVs removed because they map to no candidate MAG: {(~mapped_to_candidate_mag).sum()}")
print(f"Mapped ASVs absent from abundance table: {(~C_mapped.index.isin(Y_full.index)).sum()}")
print(f"Abundance-table ASVs outside the candidate-MAG system: {(~Y_full.index.isin(C_mapped.index)).sum()}")
print(f"ASVs used for estimation: {len(common_asvs)}")

C = C_mapped.loc[common_asvs].copy()
Y = Y_full.loc[common_asvs].copy()
Y = Y.loc[C.index]

if not C.index.equals(Y.index):
    raise RuntimeError("ASV row alignment failed.")

# Every candidate MAG must remain observable after ASV matching.
zero_mag = ~C.ne(0).any(axis=0)
if zero_mag.any():
    raise ValueError(
        "Candidate MAGs with no ASV after matching: "
        f"{C.columns[zero_mag].tolist()}"
    )


ASVs removed because they map to no candidate MAG: 0
Mapped ASVs absent from abundance table: 0
Abundance-table ASVs outside the candidate-MAG system: 0
ASVs used for estimation: 2


## 4. Remove completely missing samples/time points

A column containing only `NaN` is excluded before estimation and restored afterward. Partial missingness is rejected because removing individual ASVs can change rank and identifiability for that sample.


In [4]:
all_missing = Y.isna().all(axis=0)
partial_missing = Y.isna().any(axis=0) & ~all_missing

if partial_missing.any():
    raise ValueError(
        "Partial missingness is not supported. Affected samples/time points: "
        f"{Y.columns[partial_missing].tolist()}"
    )

Y_observed = Y.loc[:, ~all_missing].copy()
if Y_observed.shape[1] == 0:
    raise ValueError("No observed samples/time points remain for estimation.")

missing_columns = Y.columns[all_missing]
print(f"Completely missing samples/time points excluded before fitting: {len(missing_columns)}")
if len(missing_columns):
    print(missing_columns.tolist())
print(f"Observed samples/time points used for estimation: {Y_observed.shape[1]}")


Completely missing samples/time points excluded before fitting: 1
['T4']
Observed samples/time points used for estimation: 4


## 5. Estimate MAG abundance

In [5]:
X_observed = estimate_mag_abundance(
    copy_number_matrix=C,
    asv_abundance=Y_observed,
    lambda_penalty=1e-8,
    epsilon=1e-4,
    delta=1e-4,
    max_iter=50,
    tol=1e-6,
)

X_observed.iloc[:5, :5]


,T1,T2,T3,T5
MAG1,9.999841,19.999682,29.999522,39.999362
MAG2,1.999947,3.999895,5.999842,7.999790
MAG3,7.000106,14.000211,21.000317,28.000422


## 6. Restore the original sample/time-point axis and save


In [6]:
X = X_observed.reindex(columns=Y.columns)
X.to_csv(OUTPUT_CSV, index_label="MAG")

print(f"Saved: {OUTPUT_CSV}")
print(f"Output matrix: {X.shape[0]} MAGs × {X.shape[1]} samples/time points")
X.iloc[:5, :5]


Saved: mag_abundance.csv
Output matrix: 3 MAGs × 5 samples/time points


,T1,T2,T3,T4,T5
MAG1,9.999841,19.999682,29.999522,NaN,39.999362
MAG2,1.999947,3.999895,5.999842,NaN,7.999790
MAG3,7.000106,14.000211,21.000317,NaN,28.000422
